# Archived experiment notebook

Source preserved for review. Private outputs, attachments, and execution counts are removed in this public copy; original AWS notebooks remain untouched. This copy is not evidence of a fresh execution. Use the public Research Review for aggregate results. Private input contracts and artifacts are required to reproduce this historical experiment.


# 23 · Matched goal-feature ablation

**Two models, no sweep or ensemble.** Both receive the same group-separated, forecast-conditioned and normalized relationship interface. Cartesian control zeros only the six goal values, retaining their masks. The goal arm receives their actual observed history. A comparison to the old encoder changes architecture too and is not a six-feature attribution.

In [ ]:
from pathlib import Path
import json, sys, subprocess, signal, os
import plotly.io as pio
KIT = Path('/home/sagemaker-user/nfl_feature_round10')
OUT = Path('/home/sagemaker-user/nfl-feature-round10-results')
PY = Path('/home/sagemaker-user/nfl-player-trajectory/.venv/bin/python')
if not KIT.is_dir() or not PY.is_file():
    raise FileNotFoundError('Use the existing NFL space and upload/extract the Round 10 kit.')
sys.path.insert(0, str(KIT))
import visuals
pio.renderers.default = 'plotly_mimetype'
def run(stage, *options):
    child = subprocess.Popen([str(PY),str(KIT/'run_round.py'),stage,*options],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
        start_new_session=True)
    try:
        for line in child.stdout:
            print(line, end='')
        code=child.wait()
    except KeyboardInterrupt:
        os.killpg(child.pid, signal.SIGINT)
        try: child.wait(timeout=8)
        except subprocess.TimeoutExpired:
            os.killpg(child.pid, signal.SIGTERM)
            try: child.wait(timeout=4)
            except subprocess.TimeoutExpired: os.killpg(child.pid, signal.SIGKILL);child.wait()
        raise
    if code:
        raise RuntimeError(f'{stage} stopped ({code}). Preserve checkpoints and export the report. Do not change settings.')
def show(fig, name):
    visuals.save(fig, OUT, name).show()


## Offline pinned runtime and synthetic model/recovery tests
Require `grouped_runtime_ready`. No online installation is enabled. A missing cache or environment mismatch is a stop: return the report.

In [ ]:
run('runtime')

## Training-only speed and wiring gate
Require `grouped_profile_passed`. Sixteen disposable optimizer steps test finite, nonzero local goal-input gradients and a conservative per-arm projection no greater than 300 seconds. Gradient magnitude is not a predictive acceptance criterion. Do not increase budgets or instance size if it stops.

In [ ]:
run('profile')

## Fixed Cartesian control
Require `grouped_arm_complete`. Up to 24 epochs, checkpoints every 25 steps; no evaluation inputs are opened during fitting.

In [ ]:
run('train','--arm','cartesian')

## Matched goal-frame treatment
Same initialization and optimizer exposure. Require `grouped_arm_complete`. Do not select an intermediate checkpoint.

In [ ]:
run('train','--arm','goal')

## Evaluate the completed pair
One prespecified comparison: at least 1% gain and a paired-game 95% upper RMSE-difference bound below zero. To justify another fold, goal must also be no worse than the preserved tree on these rows. These are reused games, not independent confirmation.

In [ ]:
run('evaluate')
s=json.loads((OUT/'summary.json').read_text())
print(json.dumps({'metrics':s['metrics'],'contrast':s['contrast'],'ready_for_later_fold':s['ready_for_later_fold']},indent=2))
show(visuals.learning(OUT),'training_objective')
show(visuals.metrics(OUT),'matched_rmse')
show(visuals.interval(OUT),'paired_game_interval')
show(visuals.horizons(OUT),'horizon_diagnostics')

## Fresh-process replay and export
Require `grouped_goal_replay_exact` and `new_optimizer_steps: 0`. Missing models or predictions are not trained/created by replay. Export the report and stop this milestone even if it passes.

In [ ]:
run('replay')
run('report')
print(OUT/'nfl_feature_round10_report.zip')